# Atividade 2 — Camada Silver

Realização de limpeza, tradução e tipagem. A camada Bronze não é alterada, apenas extraimos as informações e salvamos na camada silver.

## 1. Preparação e funções auxiliares

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

def salvar_silver(df, tabela):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))

def manter_versao_mais_recente(df, chaves):
    payload = [c for c in df.columns if c not in chaves + ["ingestion_datetime"]]
    criterios = [F.col("ingestion_datetime").desc_nulls_last()]
    if payload:
        criterios.append(F.to_json(F.struct(*[F.col(c) for c in payload])).desc_nulls_last())
    janela = Window.partitionBy(*chaves).orderBy(*criterios)
    return (df.withColumn("_ordem", F.row_number().over(janela))
              .filter(F.col("_ordem") == 1)
              .drop("_ordem"))

AUSENCIAS = ["", "unknown", "n/a", "na", "null", "none", "not available",
             "not informed", "não informado", "nan", "-", "--"]

def numero_limpo(coluna):
    """Normaliza número com vírgula/ponto decimal ou separador de milhar."""
    original = F.trim(F.lower(coluna.cast("string")))
    ausente = original.isNull() | original.isin(AUSENCIAS)
    texto = F.regexp_replace(original, r"[^0-9,.-]", "")
    tem_virgula = texto.contains(",")
    tem_ponto = texto.contains(".")
    # Quando os dois aparecem, o separador mais à direita é considerado decimal.
    dist_virgula = F.locate(",", F.reverse(texto))
    dist_ponto = F.locate(".", F.reverse(texto))
    ambos_virgula_decimal = F.regexp_replace(F.regexp_replace(texto, r"\.", ""), ",", ".")
    ambos_ponto_decimal = F.regexp_replace(F.regexp_replace(texto, ",", ""), r"\.(?=.*\.)", "")
    somente_virgula = F.when(
        texto.rlike(r"^-?\d{1,3}(,\d{3})+$"),
        F.regexp_replace(texto, ",", "")
    ).otherwise(F.regexp_replace(F.regexp_replace(texto, r",(?=.*[,])", ""), ",", "."))
    somente_ponto = F.when(
        texto.rlike(r"^-?\d{1,3}(\.\d{3})+$"),
        F.regexp_replace(texto, r"\.", "")
    ).otherwise(F.regexp_replace(F.regexp_replace(texto, r"\.(?=.*\.)", ""), ",", "."))
    canonico = (F.when(tem_virgula & tem_ponto,
                       F.when(dist_virgula < dist_ponto, ambos_virgula_decimal)
                        .otherwise(ambos_ponto_decimal))
                 .when(tem_virgula, somente_virgula)
                 .otherwise(somente_ponto))
    valido = canonico.rlike(r"^-?\d{1,15}(\.\d{1,10})?$")
    decimal = F.when((~ausente) & valido, canonico.cast(DecimalType(38, 10)))
    return decimal

def numero_double(coluna):
    return numero_limpo(coluna).cast("double")


## 2. Série diária

In [0]:
cotacao_bruta = manter_versao_mais_recente(
    spark.table("bronze.tb_cotacao_dolar"), ["dataHoraCotacao"]
)

cotacoes = (cotacao_bruta
    .withColumn("data_cotacao", F.to_date(F.substring("dataHoraCotacao", 1, 10)))
    .withColumn("cotacao_compra", numero_double(F.col("cotacaoCompra")))
    .filter(F.col("data_cotacao").isNotNull() & (F.col("cotacao_compra") > 0)))

janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc_nulls_last())
cotacao_diaria = (cotacoes
    .withColumn("_posicao", F.row_number().over(janela_dia))
    .filter(F.col("_posicao") == 1)
    .select("data_cotacao", "cotacao_compra"))

limites = cotacao_diaria.agg(
    F.min("data_cotacao").alias("data_inicial"),
    F.max("data_cotacao").alias("data_final")
).first()
if limites["data_inicial"] is None:
    raise ValueError("Não há cotações válidas em bronze.tb_cotacao_dolar.")

calendario = (spark.range(1)
    .select(F.explode(F.sequence(
        F.lit(limites["data_inicial"]), F.lit(limites["data_final"]),
        F.expr("INTERVAL 1 DAY")
    )).alias("data_cotacao")))

janela_preenchimento = Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
dolar_silver = (calendario
    .join(cotacao_diaria, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_preenchimento))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida"))

salvar_silver(dolar_silver, "silver.tb_cotacao_dolar")
display(dolar_silver.orderBy("data_cotacao"))

taxa_usd_brl = (dolar_silver
    .orderBy(F.col("data_cotacao").desc())
    .select("cotacao_compra").first()["cotacao_compra"])
print(f"Cotação usada no cálculo: {taxa_usd_brl}")


## 3. Metadados e status

In [0]:
filmes_origem = manter_versao_mais_recente(
    spark.table("bronze.tb_movies_info").filter(F.col("id").isNotNull()), ["id"]
)

status_normalizado = F.trim(F.regexp_replace(
    F.lower(F.col("status").cast("string")), r"[^a-z]+", " "
))
mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "rumoured": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",
}
status_pt = F.lit("Não Informado")
for origem, destino in mapa_status.items():
    status_pt = F.when(status_normalizado == origem, F.lit(destino)).otherwise(status_pt)

formatos_data = [
    "yyyy-MM-dd", "yyyy-M-d", "yyyy/MM/dd", "yyyy/M/d",
    "d/M/yyyy", "dd/MM/yyyy", "M/d/yyyy", "MM/dd/yyyy",
    "d-M-yyyy", "dd-MM-yyyy", "M-d-yyyy", "MM-dd-yyyy",
    "yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss",
    "d MMM yyyy", "dd MMM yyyy", "MMM d, yyyy", "MMMM d, yyyy",
    "d MMMM yyyy", "yyyyMMdd"
]
data_texto = F.trim(F.col("release_date").cast("string"))
data_convertida = F.coalesce(*[
    F.to_date(F.expr(f"try_to_timestamp(trim(release_date), '{fmt}')"))
    for fmt in formatos_data
])

duracao_decimal = numero_limpo(F.col("runtime"))
duracao_int = F.when(
    (duracao_decimal >= 0) & (duracao_decimal <= 2147483647)
    & (duracao_decimal == F.floor(duracao_decimal)),
    duracao_decimal.cast("int")
)
info_filmes = (filmes_origem.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    data_convertida.alias("data_lancamento"),
    duracao_int.alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    status_pt.alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao")
).withColumn("ano_lancamento", F.year("data_lancamento")))

salvar_silver(info_filmes, "silver.tb_info_filmes")
display(info_filmes.groupBy("status_filme").count().orderBy(F.desc("count")))


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5718433261581142>, line 1
----> 1 filmes_origem = manter_versao_mais_recente(
      2     spark.table("bronze.tb_movies_info").filter(F.col("id").isNotNull()), ["id"]
      3 )
      5 status_normalizado = F.trim(F.regexp_replace(
      6     F.lower(F.col("status").cast("string")), r"[^a-z]+", " "
      7 ))
      8 mapa_status = {
      9     "released": "Lançado",
     10     "post production": "Pós-Produção",
   (...)     16     "cancelled": "Cancelado",
     17 }

NameError: name 'manter_versao_mais_recente' is not defined

## 4. Valores em Dólar e Real

In [0]:
def valor_positivo(coluna):
    valor = numero_limpo(coluna).cast(DecimalType(18, 2))
    return F.when(valor > 0, valor)

financeiro_origem = manter_versao_mais_recente(
    spark.table("bronze.tb_movies_financials").filter(F.col("id").isNotNull()), ["id"]
)
financeiro = financeiro_origem.select(
    F.col("id").cast("string").alias("id_filme"),
    valor_positivo(F.col("budget")).alias("orcamento_usd"),
    valor_positivo(F.col("revenue")).alias("receita_usd")
)

taxa_decimal = F.lit(float(taxa_usd_brl)).cast(DecimalType(18, 6))
financeiro = (financeiro
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * taxa_decimal).cast(DecimalType(18, 2)))
    .withColumn("receita_brl", (F.col("receita_usd") * taxa_decimal).cast(DecimalType(18, 2)))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast(DecimalType(18, 2)))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast(DecimalType(18, 2)))
    .withColumn("margem_lucro_pct",
        F.when(F.col("orcamento_usd") > 0,
            F.round(F.col("lucro_usd") / F.col("orcamento_usd") * 100, 2))))

salvar_silver(financeiro, "silver.tb_financeiro_filmes")
display(financeiro.limit(10))


## 5. Notas e contagens


In [0]:
metricas_origem = manter_versao_mais_recente(
    spark.table("bronze.tb_movies_metrics").filter(F.col("id").isNotNull()), ["id"]
)

def nota_0_a_10(coluna):
    valor = numero_double(coluna)
    return F.when((valor >= 0) & (valor <= 10), valor)

def contagem_inteira_nao_negativa(coluna):
    valor_decimal = numero_limpo(coluna)
    texto = F.trim(coluna.cast("string"))
    inteiro = numero_limpo(coluna)
    return F.when(
        (inteiro >= 0) & (inteiro <= 2147483647) & (inteiro == F.floor(inteiro)),
        inteiro.cast("int")
    )

popularidade_valor = numero_double(F.col("popularity"))
metricas = metricas_origem.select(
    F.col("id").cast("string").alias("id_filme"),
    F.when(popularidade_valor >= 0, popularidade_valor).alias("popularidade"),
    nota_0_a_10(F.col("vote_average")).alias("nota_media_tmdb"),
    contagem_inteira_nao_negativa(F.col("vote_count")).alias("qtd_votos_tmdb"),
    nota_0_a_10(F.col("averageRating")).alias("nota_media_imdb"),
    contagem_inteira_nao_negativa(F.col("numVotes")).alias("qtd_votos_imdb")
)

salvar_silver(metricas, "silver.tb_metricas_engajamento")
display(metricas.describe())


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5718433261581146>, line 1
----> 1 metricas_origem = manter_versao_mais_recente(
      2     spark.table("bronze.tb_movies_metrics").filter(F.col("id").isNotNull()), ["id"]
      3 )
      5 def nota_0_a_10(coluna):
      6     valor = numero_double(coluna)

NameError: name 'manter_versao_mais_recente' is not defined

## 6. Avaliações

In [0]:
avaliacoes_origem = spark.table("bronze.tb_movies_reviews")
avaliacoes_unicas = avaliacoes_origem.dropDuplicates(["id", "nome", "nota", "comentario"])
nota_avaliacao = numero_double(F.col("nota"))
comentario_limpo = F.trim(F.col("comentario").cast("string"))

avaliacoes = avaliacoes_unicas.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.when((nota_avaliacao >= 0) & (nota_avaliacao <= 10), nota_avaliacao).alias("nota_usuario"),
    F.when(comentario_limpo.isNull() | (comentario_limpo == ""), F.lit("Sem comentário"))
     .otherwise(F.col("comentario")).alias("comentario_usuario")
)

salvar_silver(avaliacoes, "silver.tb_avaliacoes_usuarios")
display(avaliacoes.limit(10))


## 7. Um gênero por filme

In [0]:
creditos_origem = manter_versao_mais_recente(
    spark.table("bronze.tb_credits_and_tags").filter(F.col("id").isNotNull()), ["id"]
)

def explodir_lista(df, coluna_origem, nome_valor="valor"):
    itens = F.split(F.regexp_replace(F.col(coluna_origem).cast("string"), r"[;]", ","), r"[,]")
    return (df.select(F.col("id").cast("string").alias("id_filme"), F.explode_outer(itens).alias(nome_valor))
              .withColumn(nome_valor, F.trim(F.col(nome_valor))))

generos_validos = [
    "action", "adventure", "animation", "comedy", "crime", "documentary",
    "drama", "family", "fantasy", "history", "horror", "music", "mystery",
    "romance", "science fiction", "tv movie", "thriller", "war", "western"
]
generos = (explodir_lista(creditos_origem, "genres")
    .withColumn("_chave", F.lower(F.regexp_replace(F.col("valor"), r"\s+", " ")))
    .filter(F.col("_chave").isin(generos_validos))
    .select("id_filme", F.initcap("_chave").alias("genero"))
    .dropDuplicates(["id_filme", "genero"]))

salvar_silver(generos, "silver.tb_generos")
display(generos.groupBy("genero").count().orderBy(F.desc("count")))


## 8. Entidades unificadas


In [0]:
colunas_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

partes_entidades = []
for coluna, tipo in colunas_entidades.items():
    parte = (explodir_lista(creditos_origem, coluna)
        .withColumn("valor", F.regexp_replace(F.col("valor"), r"^[\[\]{}\"']+|[\[\]{}\"']+$", ""))
        .withColumn("valor", F.trim(F.col("valor")))
        .filter(F.col("valor").isNotNull())
        .filter((F.length("valor") > 1) & (F.length("valor") <= 100))
        .filter(F.col("valor").rlike(r"[A-Za-zÀ-ÿ]"))
        .filter(~F.lower(F.col("valor")).isin(AUSENCIAS))
        .select(
            "id_filme",
            F.initcap(F.col("valor")).alias("nome_entidade"),
            F.lit(tipo).alias("tipo_entidade")
        ))
    partes_entidades.append(parte)

pessoas_empresas = partes_entidades[0]
for parte in partes_entidades[1:]:
    pessoas_empresas = pessoas_empresas.unionByName(parte)
pessoas_empresas = pessoas_empresas.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

salvar_silver(pessoas_empresas, "silver.tb_pessoas_empresas")
display(pessoas_empresas.groupBy("tipo_entidade").count().orderBy("tipo_entidade"))


## 9. Verificações finais

In [0]:
tabelas_silver = [
    "tb_cotacao_dolar", "tb_info_filmes", "tb_financeiro_filmes",
    "tb_metricas_engajamento", "tb_avaliacoes_usuarios",
    "tb_generos", "tb_pessoas_empresas"
]
for nome in tabelas_silver:
    tabela = f"silver.{nome}"
    print(f"{tabela}: {spark.table(tabela).count()} linhas")
